In [53]:
import os 
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is set")
else:
    raise ValueError("OPENAI_API_KEY is not set")

OPENAI_API_KEY is set


In [54]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

loader = DirectoryLoader(
    "./IKSPL_Emp_Policies",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)
docs = loader.load()

In [55]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

In [56]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [57]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2142.64it/s]


In [58]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Create LLM
llm = ChatOpenAI(model="gpt-5-mini",temperature=0.9)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a company policy assistant.
Answer ONLY using the provided context.
If not found, say "I don't know".

Context:
{context}
"""),
    ("human", "{question}")   # ✅ IMPORTANT
])

# Step 2: Create QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_template}
)

In [59]:
print(qa_chain.input_keys)

['query']


In [61]:
response = qa_chain.invoke({
    'query': "What is leave policy?"
})
print(response["result"])

The leave policy (Leave & Attendance Policy) at iScholar Knowledge Services Pvt Ltd:

- Objective: To help employees maintain a healthy work–life balance by providing leave for sickness, recuperation, emergencies, personal work, rest/recreation and social obligations.  
- Applicability: All regular, permanent staff of IKSPL.  
- Leave year: Calendar year (January–December).  
- Entitlement: 24 days per year total — Casual Leave (CL) 6 days, Sick Leave (SL) 6 days, Earned Leave (EL) 12 days. Employees appointed during the year get leave on a pro‑rata basis.  
- Grant of leave: Subject to work exigencies and at the discretion of the Reporting Manager/management.  
- Leave calculation rules: Intervening Saturdays, Sundays and Company holidays prefixed, suffixed or in between leave are not counted. If leave is Loss of Pay, intervening Saturdays, Sundays and Company holidays are counted as Loss of Pay.
